In [1]:
library('DESeq2')
library('BiocParallel')
register(MulticoreParam(4))

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, saveRDS, setdiff,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb



In [68]:
# Snakemake I/O
count_file <- "../results/count_matrix.tsv"
coldata_file <- "../results/coldata.tsv"
cell_line <- "H1299"
perturbation <- "RUBY"

In [63]:
# Load data
df <- read.csv(file = count_file, header = TRUE, sep = "\t", check.names = FALSE)
countData <- df[, c(2, 6:ncol(df))]
rownames(countData) <- countData[[1]]
countData[[1]] <- NULL
colData <- read.csv(file = coldata_file, row.names=1, sep="\t")
colData$perturbation <- factor(colData$perturbation)
colData$perturbation <- droplevels(colData$perturbation)
colData$perturbation <- relevel(colData$perturbation, ref = "NT")  # set NT as reference
colData$guide        <- factor(colData$guide)

In [64]:
# Subset to the CAS genotype
count <- countData[, colData$genotype == "CAS"]
col   <- droplevels(colData[colData$genotype == "CAS", ])

# Subset to the specified cell line
count <- count[, col$line == cell_line]
col   <- droplevels(col[col$line == cell_line, ])

In [17]:
all(colnames(count) == rownames(col))  # sanity check

[1] TRUE

In [ ]:
# Create DESeq2 dataset
dds <- DESeqDataSetFromMatrix(countData = count,
                              colData = col,
                              design = ~ perturbation)
dds <- dds[rowSums(counts(dds)) >= 10, ]  # min counts per gene

In [7]:
# Run DESeq
dds <- DESeq(dds)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

-- replacing outliers and refitting for 1 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)

estimating dispersions

fitting model and testing



In [20]:
  # Construct the coefficient name
coef_name <- paste0("perturbation_", perturbation, "_vs_NT")
res <- results(dds, name = coef_name)
res_shr <- lfcShrink(dds, coef = coef_name, type = "apeglm")
# Order by p-value
resOrdered <- res_shr[order(res_shr$pvalue), ]

using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895



In [21]:
resultsNames(dds)  # lists the coefficients

[1] "Intercept"                 "perturbation_FBL_vs_NT"   
 [3] "perturbation_FTL_vs_NT"    "perturbation_LINC_vs_NT"  
 [5] "perturbation_MOCK_vs_NT"   "perturbation_PPIA_vs_NT"  
 [7] "perturbation_RUBY_vs_NT"   "perturbation_SNHG15_vs_NT"
 [9] "perturbation_SNHG16_vs_NT" "perturbation_VIM_vs_NT"

In [24]:
# count number of sig genes
sum(resOrdered$padj < 0.05, na.rm=TRUE)

[1] 3053

# Guide-wise

In [69]:
# Load data
df <- read.csv(file = count_file, header = TRUE, sep = "\t", check.names = FALSE)
countData <- df[, c(2, 6:ncol(df))]
rownames(countData) <- countData[[1]]
countData[[1]] <- NULL
colData <- read.csv(file = coldata_file, row.names=1, sep="\t")
colData$perturbation <- factor(colData$perturbation)
colData$perturbation <- droplevels(colData$perturbation)
colData$perturbation <- relevel(colData$perturbation, ref = "NT")  # set NT as reference
colData$guide        <- factor(colData$guide)

# Subset to the CAS genotype
count <- countData[, colData$genotype == "CAS"]
col   <- droplevels(colData[colData$genotype == "CAS", ])

# Subset to the specified cell line
count <- count[, col$line == cell_line]
col   <- droplevels(col[col$line == cell_line, ])

col <- col[col$perturbation != "MOCK", ]
count <- count[, rownames(col)]
# reset factors
col$perturbation <- droplevels(col$perturbation)
col$guide        <- droplevels(col$guide)

all(colnames(count) == rownames(col))  # sanity check

[1] TRUE

In [70]:
# Set NAN entries of col$guide to "no_guide"


In [71]:
# Create DESeq2 dataset
# Guide is not a global factor (MALAT1 g1 is not comparable to VIM g1 for instance)
# So instead we create a combined factor KO_guide, i.e. guides are nested within perturbations
count <- count[, rownames(col)]
col$KO_guide <- interaction(col$perturbation, col$guide, drop=TRUE, sep="_")
col$KO_guide <- relevel(col$KO_guide, ref = "NT_1")
dds <- DESeqDataSetFromMatrix(countData = count,
                              colData = col,
                              design = ~ KO_guide)
dds <- dds[rowSums(counts(dds)) >= 10, ]  # min counts per gene

In [ ]:
# Run DESeq
dds <- DESeq(dds)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [57]:
# Let's export ruby and NT tests
names = resultsNames(dds)
for (i in 1:4){
    for (j in 1:1){
        coef_name <- paste0("KO_guide_RUBY_", i, "_vs_NT_", j)
        res <- results(dds, name = coef_name)
        write.csv(as.data.frame(res), file = paste0("../results/deseq2_results_per_guide/", cell_line, "_RUBY", i, "_vs_NT", j, ".csv"))
    }
}

In [55]:
coef_name <- "KO_guide_NT_2_vs_NT_1"
res <- results(dds, name = coef_name)
write.csv(as.data.frame(res), file = paste0("../results/deseq2_results_per_guide/",cell_line,"_NT_2_vs_NT_1.csv"))

coef_name <- "KO_guide_NT_3_vs_NT_1"
res <- results(dds, name = coef_name)
write.csv(as.data.frame(res), file = paste0("../results/deseq2_results_per_guide/",cell_line,"_NT_3_vs_NT_1.csv"))

In [61]:
# Run all control guides (NTs and RUBYs) against each NT respectively.
for (j in 1:3){
    reference = paste0("NT_", j)
    count <- count[, rownames(col)]
    col$KO_guide <- interaction(col$perturbation, col$guide, drop=TRUE, sep="_")
    col$KO_guide <- relevel(col$KO_guide, ref = reference)
    dds <- DESeqDataSetFromMatrix(countData = count,
                                colData = col,
                                design = ~ KO_guide)
    dds <- dds[rowSums(counts(dds)) >= 10, ]  # min counts per gene
    # Run DESeq
    dds <- DESeq(dds)

    # Let's export ruby and NT tests
    names = resultsNames(dds)
    for (i in 1:4){
        coef_name <- paste0("KO_guide_RUBY_", i, "_vs_NT_", j)
        res <- results(dds, name = coef_name)
        write.csv(as.data.frame(res), file = paste0("../results/deseq2_results_per_guide/", cell_line, "_RUBY", i, "_vs_NT", j, ".csv"))
    }
    for (i in 1:3){
        if (i != j){
            coef_name <- paste0("KO_guide_NT_", i, "_vs_NT_", j)
            res <- results(dds, name = coef_name)
            write.csv(as.data.frame(res), file = paste0("../results/deseq2_results_per_guide/", cell_line, "_NT", i, "_vs_NT", j, ".csv"))
        }
    }
}

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [60]:
coef_name

[1] "KO_guide_NT_4_vs_NT_1"